In [2]:
from pyspark.sql import functions as sf
from pyspark.sql import window as sw
from pyspark.sql import types as sdt
from pyspark.sql import SparkSession
from datetime import datetime

LOCAL_WAREHOUSE_PATH = "/data/data_files/iceberg/iceberg_warehouse"
STG_WAREHOUSE_PATH = "/data/data_files/iceberg/iceberg_staging_warehouse"
RPT_WAREHOUSE_PATH = "/data/data_files/iceberg/WideWorldImportersDW"

MSSQL_JAR = "C:/data/spark/jars/mssql-jdbc-12.6.5.jre11.jar"

CATALOG_NAME = "local"
STG_CATALOG_NAME = "staging"
WH_CATALOG_NAME = "reporting"

staging_table_name = "staging.Integration.employee_Staging"
wh_table_name = "reporting.dimension.Employees"

spark = SparkSession.builder \
    .appName("Iceberg Setup") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}.warehouse", f"file:///{LOCAL_WAREHOUSE_PATH}") \
    .config(f"spark.sql.catalog.{STG_CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{STG_CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{STG_CATALOG_NAME}.warehouse", f"file:///{STG_WAREHOUSE_PATH}") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}.warehouse", f"file:///{RPT_WAREHOUSE_PATH}") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.jars", f"file:///{MSSQL_JAR}") \
    .config("spark.driver.extraClassPath", MSSQL_JAR) \
    .config("spark.executor.extraClassPath", MSSQL_JAR) \
    .getOrCreate()


spark.catalog.setCurrentCatalog("local")

spark



In [3]:
import re
from pyspark.sql.utils import AnalysisException

# --- Config (re-use your existing vars) ---
JDBC_DRIVER = "com.microsoft.sqlserver.jdbc.SQLServerDriver"
JDBC_URL = "jdbc:sqlserver://AshishPC:1433;databaseName=WideWorldImportersDW;encrypt=false;trustServerCertificate=true;"
DB_USER = "sa"
DB_PASSWORD = "sa123#"

# catalog where Iceberg tables will be written
WH_CATALOG_NAME = "reporting"

# Read list of base tables (you already did this; re-run if needed)
tables_subq = ("(Select TABLE_SCHEMA, TABLE_NAME "
               "From INFORMATION_SCHEMA.TABLES "
               "Where TABLE_TYPE = 'BASE TABLE') AS TABLES_SUBQ")

tables_df = spark.read.format("jdbc") \
    .option("driver", JDBC_DRIVER) \
    .option("url", JDBC_URL) \
    .option("dbtable", tables_subq) \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .option("fetchsize", "1000") \
    .load()

tables = tables_df.collect()  # small resultset, safe to collect

# helper to sanitize target table names (no spaces, no special chars)
def sanitize(name: str) -> str:
    name = name.strip()
    # replace whitespace and non-alphanumeric by underscore, collapse multiple underscores
    name = re.sub(r'[^0-9a-zA-Z]+', '_', name)
    name = re.sub(r'_{2,}', '_', name)
    return name.lower()

# Create logging lists
success = []
failures = []

# Loop through tables
for row in tables:
    src_schema = row['TABLE_SCHEMA']
    src_table = row['TABLE_NAME']
    if src_schema is None or src_table is None:
        continue

    # build safe fully-qualified source; use bracket quoting to support spaces / special chars
    src_fq = f"[{src_schema}].[{src_table}]"
    src_subq = f"(SELECT * FROM {src_fq}) AS src"

    # determine target namespace and table name in Iceberg:
    # keep schema as namespace, sanitized table name
    target_namespace = sanitize(src_schema)   # e.g. "integration" -> "integration"
    target_table_name = sanitize(src_table)  # e.g. "ETL Cutoff" -> "etl_cutoff"
    iceberg_full_name = f"{WH_CATALOG_NAME}.{target_namespace}.{target_table_name}"

    print(f"\nProcessing source: {src_fq}  ->  {iceberg_full_name}")

    try:
        # Create namespace if it doesn't exist
        try:
            spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {WH_CATALOG_NAME}.{target_namespace}")
            print(f"Ensured namespace: {WH_CATALOG_NAME}.{target_namespace}")
        except Exception as e_ns:
            # non-fatal; log and continue
            print(f"Warning: could not create namespace {WH_CATALOG_NAME}.{target_namespace}: {e_ns}")

        # Read table from SQL Server
        df = spark.read.format("jdbc") \
            .option("driver", JDBC_DRIVER) \
            .option("url", JDBC_URL) \
            .option("dbtable", src_subq) \
            .option("user", DB_USER) \
            .option("password", DB_PASSWORD) \
            .option("fetchsize", "1000") \
            .load()

        print(f"Read table {src_fq}: rows={df.rdd.getNumPartitions()} partitions (schema below)")
        df.printSchema()

        # If you want to preview first N rows, uncomment:
        # df.show(5, truncate=False)

        # Write -> Iceberg. Choose mode: "append" to keep existing data, "overwrite" to recreate.
        write_mode = "append"

        # If table doesn't exist, using saveAsTable will register it. For safety, use saveAsTable for create,
        # but to do this conditionally we check existence.
        table_exists = False
        try:
            spark.sql(f"DESCRIBE TABLE {iceberg_full_name}")
            table_exists = True
        except AnalysisException:
            table_exists = False

        if table_exists:
            print(f"Table exists: {iceberg_full_name} -> appending data")
            df.write.format("iceberg").mode("append").save(iceberg_full_name)
        else:
            print(f"Table does not exist: {iceberg_full_name} -> creating table and writing data")
            # Create table by saveAsTable (registers in the reporting catalog)
            df.write.format("iceberg").mode("overwrite").saveAsTable(iceberg_full_name)
            # if you'd rather create empty table first with partitions, you could use CREATE TABLE DDL

        # Quick verification (count few rows)
        try:
            cnt = spark.sql(f"SELECT count(*) as c FROM {iceberg_full_name}").collect()[0]['c']
            print(f"Wrote data to {iceberg_full_name} (count={cnt})")
        except Exception as e_v:
            print(f"Warning: couldn't verify count for {iceberg_full_name}: {e_v}")

        success.append(iceberg_full_name)

    except Exception as e:
        print(f"ERROR processing {src_fq} -> {iceberg_full_name}: {e}")
        failures.append((src_fq, str(e)))
        # continue to next table

# Summary
print("\n===== SUMMARY =====")
print(f"Succeeded: {len(success)} tables")
for t in success[:20]:
    print("  -", t)
print(f"Failed: {len(failures)} tables")
for src, err in failures:
    print("  -", src, "->", err)



Processing source: [Integration].[ETL Cutoff]  ->  reporting.integration.etl_cutoff
Ensured namespace: reporting.integration
Read table [Integration].[ETL Cutoff]: rows=1 partitions (schema below)
root
 |-- Table Name: string (nullable = true)
 |-- Cutoff Time: timestamp (nullable = true)

Table exists: reporting.integration.etl_cutoff -> appending data
Wrote data to reporting.integration.etl_cutoff (count=28)

Processing source: [Integration].[Lineage]  ->  reporting.integration.lineage
Ensured namespace: reporting.integration
Read table [Integration].[Lineage]: rows=1 partitions (schema below)
root
 |-- Lineage Key: integer (nullable = true)
 |-- Data Load Started: timestamp (nullable = true)
 |-- Table Name: string (nullable = true)
 |-- Data Load Completed: timestamp (nullable = true)
 |-- Was Successful: boolean (nullable = true)
 |-- Source System Cutoff Time: timestamp (nullable = true)

Table exists: reporting.integration.lineage -> appending data
Wrote data to reporting.integ